# Wan text-to-video server for `Youtube_video_automation`

Backing endpoint for `tools/colab_video.py` / `scripts/produce_reel_clip.py`, and the main
pipeline's AI_VIDEO beats when `AI_VIDEO_PROVIDER=colab` (or `auto`) is set in `.env`.

## Everything here is shaped by one constraint

A free Colab T4 has **~12.7 GB of system RAM** and **~14.6 GB of usable VRAM**. Wan2.1's parts do
not fit in either one together:

| component | size |
|---|---|
| UMT5-XXL text encoder | 22.7 GB fp32 on disk → **~11.4 GB** fp16 in memory |
| transformer (1.3B) + VAE (kept fp32) | **~2.9 GB** |

Three arrangements crash, and **two of them leave no traceback at all** — the kernel is `SIGKILL`ed
by the OS and the Jupyter log only says `restarting kernel (1/5)`:

1. **`enable_model_cpu_offload()`** keeps the encoder in *system RAM* — 11.4 of 12.7 GB. Killed.
2. **`HF_XET_HIGH_PERFORMANCE=1`** is a download-buffer preset (16 GB buffer, ~18 GB peak RSS)
   that HF documents as needing **64 GB of RAM**. It makes the download ~2x faster and then kills
   the kernel mid-download. Killed.
3. **Encoder and transformer both on the GPU** — 11.4 + 2.9 > 14.6 GB. This one *does* raise, as a
   CUDA OOM on the encoder's last shard.

So: the GPU holds one big model at a time, and **the two memory-hungry steps run in subprocesses**.
A child process that gets OOM-killed cannot take the kernel with it — the parent sees a non-zero
exit code, reports it clearly, and the Gradio server stays up. That is the difference between "the
session died again" and an error message naming the step.

```
kernel (Gradio server, transformer+VAE on CPU — ~3 GB, always safe)
   │
   ├─ prefetch  → child process: downloads 22.7 GB
   ├─ encode    → child process: encoder → GPU (11.4 GB) → embeddings → exits, RAM reclaimed by OS
   └─ denoise   → in-kernel: pipeline → GPU (2.9 GB) → frames → back to CPU
```

`WanPipeline.__call__` never touches `self.text_encoder`, so the pipeline is loaded with
`text_encoder=None, tokenizer=None` (diffusers skips any component passed as `None`) and driven
entirely from `prompt_embeds`.

**Do not** add `enable_model_cpu_offload()`, **do not** set `HF_XET_HIGH_PERFORMANCE` on this box,
and **do not** leave `pipe` on the GPU during encode. Those are the three crashes above.

## Model, chosen from the GPU it lands on

| Runtime | VRAM | Model | Output | Speed |
|---|---|---|---|---|
| Free **T4** | 16 GB | `Wan-AI/Wan2.1-T2V-1.3B-Diffusers` | 832x480 @ 16fps | ~6-12 min / 5 s clip |
| Pro **L4 / A100** | 24-40 GB | `Wan-AI/Wan2.2-TI2V-5B-Diffusers` | 1280x704 @ 24fps | ~5-9 min / 5 s clip |

Both Apache-2.0. **dtype** is picked from the GPU too: `bfloat16` has no hardware support before
Ampere (`sm_80`), so a T4 (`sm_75`) gets `float16` — torch accepts bf16 there but runs an emulated
slow path.

## How to use
1. `Runtime -> Change runtime type -> T4 GPU` (or L4/A100 on Colab Pro), then `Run all`.
2. The first run downloads ~23 GB. Mount Drive (cell below) if you have ~30 GB spare, and it
   becomes a one-time cost instead of per-session.
3. Wait for the last cell to print `https://xxxxxxxx.gradio.live`.
4. Put that URL in `.env` as `COLAB_VIDEO_URL=...` — **it changes every restart**.
5. Keep the tab open. Free sessions drop after ~90 min idle, ~12 h hard cap.


In [ ]:
# Wan2.1 needs diffusers >= 0.33; Wan2.2-TI2V-5B needs >= 0.35. accelerate is required for the
# device_map streaming load. sentencepiece + protobuf are for the UMT5 tokenizer; ftfy for prompt
# cleanup; imageio-ffmpeg to write the mp4.
!pip install -q -U "diffusers>=0.35.0" transformers accelerate ftfy sentencepiece protobuf \
    imageio imageio-ffmpeg gradio gradio_client psutil


## Optional but strongly recommended: persist the model cache to Drive

The text encoder alone is **22.7 GB** and Colab wipes its disk between sessions, so by default you
re-download it *every* session. Mounting Drive and pointing `HF_HOME` at it makes that a one-time
cost — later sessions start in seconds.

Needs ~30 GB of Drive space, which is more than the free 15 GB tier. **Skip this cell if you don't
have the room** — everything still works, it just re-downloads each time. (Pointing `HF_HOME` at a
Drive that runs out of space mid-download is worse than not using it.)


In [ ]:
# Optional. Run BEFORE the next cell if you want the download cached across sessions.
from google.colab import drive

drive.mount("/content/drive")


In [ ]:
import os
import shutil

# --- environment, set before anything imports huggingface_hub or torch ---

try:
    import psutil

    RAM_GB = psutil.virtual_memory().total / 1e9
except Exception:
    RAM_GB = 0.0

# HF_XET_HIGH_PERFORMANCE is a download-buffer preset (16 GB buffer, ~18 GB peak RSS) that HF
# documents as being for machines with >=64 GB of RAM. On a 12.7 GB Colab box it roughly doubles
# download speed and then gets the kernel OOM-killed mid-download, with no traceback. Only enable
# it where there is actually memory to buffer into.
if RAM_GB >= 64:
    os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
    print(f"{RAM_GB:.0f} GB RAM -> Xet high-performance download enabled.")
else:
    os.environ.pop("HF_XET_HIGH_PERFORMANCE", None)
    # Safe middle ground: more parallel range requests than the default 16 costs little memory,
    # unlike the high-performance preset's giant buffers.
    os.environ.setdefault("HF_XET_NUM_CONCURRENT_RANGE_GETS", "24")
    print(f"{RAM_GB:.0f} GB RAM -> Xet high-performance DISABLED (needs >=64 GB; it OOM-kills this box).")

# This notebook allocates and frees an 11.4 GB model repeatedly -- exactly the pattern that
# fragments the caching allocator.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

if os.path.isdir("/content/drive/MyDrive"):
    os.environ.setdefault("HF_HOME", "/content/drive/MyDrive/hf_cache")
    print(f"HF cache -> {os.environ['HF_HOME']} (persists across sessions)")
else:
    print("HF cache -> ephemeral session disk; the ~23 GB re-downloads next session.")

free_gb = shutil.disk_usage(os.environ.get("HF_HOME", "/content")).free / 1e9
print(f"Free space at cache location: {free_gb:.0f} GB" + ("  <-- TIGHT, need ~30" if free_gb < 30 else ""))

# --- GPU ---

import torch

CAPABILITY = (0, 0)
VRAM_GB = 0.0
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    CAPABILITY = (props.major, props.minor)
    VRAM_GB = props.total_memory / 1e9
    print(f"\nGPU: {props.name}  ({VRAM_GB:.0f} GB VRAM, compute capability {props.major}.{props.minor})")
    print(f"System RAM: {RAM_GB:.0f} GB")
    if CAPABILITY < (8, 0):
        print("  pre-Ampere: no bfloat16 hardware -> using float16.")
    if VRAM_GB < 20:
        print("  <20 GB VRAM -> Wan2.1-T2V-1.3B (832x480).")
        print("  Encoder (11.4 GB) and transformer (2.9 GB) will take turns on the GPU.")
    else:
        print("  >=20 GB VRAM -> Wan2.2-TI2V-5B (1280x704 @ 24fps).")
else:
    print("\nNO GPU. Wan on CPU is impractically slow (hours/clip). "
          "Runtime > Change runtime type > T4 GPU.")


def ram_report(note: str) -> None:
    """RAM is the resource that kills this notebook silently, so every big step prints it."""
    try:
        import psutil

        vm = psutil.virtual_memory()
        used = (vm.total - vm.available) / 1e9
        gpu = f", GPU {torch.cuda.memory_allocated() / 1e9:.1f} GB" if torch.cuda.is_available() else ""
        print(f"    [mem] {note}: RAM {used:.1f}/{vm.total / 1e9:.1f} GB{gpu}", flush=True)
    except Exception:  # noqa: S110 - reporting memory must never break a render
        pass


ram_report("after imports")


In [ ]:
import time

import torch
from diffusers import AutoencoderKLWan, UniPCMultistepScheduler, WanPipeline

BIG_GPU = VRAM_GB >= 20
MODEL_ID = "Wan-AI/Wan2.2-TI2V-5B-Diffusers" if BIG_GPU else "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"
DTYPE = torch.bfloat16 if CAPABILITY >= (8, 0) else torch.float16
DTYPE_NAME = "bfloat16" if CAPABILITY >= (8, 0) else "float16"  # passed to the encode subprocess
FLOW_SHIFT = 5.0 if BIG_GPU else 3.0  # Wan guidance: 3.0 for <=480p, 5.0 for 720p
MAX_SEQUENCE_LENGTH = 512             # WanPipeline.__call__'s own default; the encoder matches it

# A GPU big enough for the encoder (11.4 GB) and the transformer at once doesn't need the swap.
KEEP_PIPE_ON_GPU = VRAM_GB >= 24

DEFAULT_W, DEFAULT_H = (1280, 704) if BIG_GPU else (832, 480)
DEFAULT_FRAMES = 121 if BIG_GPU else 81   # ~5 s at 24fps / 16fps respectively
EXPORT_FPS = 24 if BIG_GPU else 16        # each model's training fps

print(f"Loading {MODEL_ID} (transformer + VAE only) in {DTYPE_NAME}...", flush=True)
_t0 = time.time()

# VAE in fp32 on purpose: Wan's VAE is numerically unstable in half precision.
vae = AutoencoderKLWan.from_pretrained(MODEL_ID, subfolder="vae", torch_dtype=torch.float32)

# text_encoder/tokenizer explicitly None -> diffusers skips loading them entirely, so the 22.7 GB
# encoder never enters this process. The encode subprocess handles it.
pipe = WanPipeline.from_pretrained(
    MODEL_ID, vae=vae, text_encoder=None, tokenizer=None, torch_dtype=DTYPE
)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config, flow_shift=FLOW_SHIFT)

# Stays on the CPU. `generate` moves it to the GPU only once the encoder subprocess has exited.
# No enable_model_cpu_offload() -- see the notebook header.
if KEEP_PIPE_ON_GPU:
    pipe.to(DEVICE)

try:
    pipe.vae.enable_tiling()
except AttributeError:
    pass

print(f"Loaded in {time.time() - _t0:.0f}s.  Output: {DEFAULT_W}x{DEFAULT_H} @ {EXPORT_FPS}fps", flush=True)
print(f"pipe resident on: {'GPU' if KEEP_PIPE_ON_GPU else 'CPU (swapped in per clip)'}", flush=True)
ram_report("pipeline loaded")


In [ ]:
import subprocess
import sys
import time

# The encoder work lives in a standalone script run as a child process.
#
# Loading an 11.4 GB model is the step most likely to be OOM-killed, and a kill inside the kernel
# takes the Gradio server with it and leaves no traceback -- that is the failure mode this notebook
# keeps hitting. In a child process the OS reclaims everything on exit, and a kill shows up here as
# a return code we can name instead of a dead session.
ENCODE_SCRIPT = "/content/wan_encode.py"

with open(ENCODE_SCRIPT, "w") as f:
    f.write('\nimport argparse\nimport torch\nfrom diffusers import WanPipeline\nfrom transformers import AutoTokenizer, UMT5EncoderModel\n\np = argparse.ArgumentParser()\np.add_argument("--model-id", required=True)\np.add_argument("--prompt", required=True)\np.add_argument("--negative", default="")\np.add_argument("--out", required=True)\np.add_argument("--dtype", default="float16")\np.add_argument("--max-len", type=int, default=512)\np.add_argument("--device", default="cuda")\na = p.parse_args()\n\ndtype = getattr(torch, a.dtype)\n\n# device_map streams shards straight to the GPU, so host RAM never holds the whole model.\ntokenizer = AutoTokenizer.from_pretrained(a.model_id, subfolder="tokenizer")\ntext_encoder = UMT5EncoderModel.from_pretrained(\n    a.model_id, subfolder="text_encoder", torch_dtype=dtype, device_map={"": a.device}\n)\n# Encode-only view: transformer/transformer_2 are optional components, vae passed as None.\npipe = WanPipeline.from_pretrained(\n    a.model_id, tokenizer=tokenizer, text_encoder=text_encoder,\n    vae=None, transformer=None, torch_dtype=dtype,\n)\nprompt_embeds, negative_embeds = pipe.encode_prompt(\n    prompt=a.prompt, negative_prompt=a.negative, do_classifier_free_guidance=True,\n    max_sequence_length=a.max_len, device=a.device, dtype=dtype,\n)\n# Saved on the CPU; the parent moves them back to the GPU for the denoise.\ntorch.save(\n    {"prompt_embeds": prompt_embeds.cpu(), "negative_prompt_embeds": negative_embeds.cpu()},\n    a.out,\n)\nprint("ENCODE_OK", flush=True)\n')


def _run_child(argv: list, label: str, timeout: int = 7200) -> None:
    """Run a child process, turning an OOM-kill into a readable error, not a dead kernel."""
    t0 = time.time()
    proc = subprocess.run(argv, capture_output=True, text=True, timeout=timeout, check=False)
    if proc.returncode == 0:
        print(f"    {label} finished in {time.time() - t0:.0f}s", flush=True)
        return

    # -9 / 137 is SIGKILL, which on Colab means the OOM killer.
    if proc.returncode in (-9, 137):
        raise RuntimeError(
            f"{label} was OOM-killed by the OS after {time.time() - t0:.0f}s. The kernel survived "
            f"because this ran in a subprocess. Restart the runtime to free RAM, or use a high-RAM "
            f"Colab Pro instance."
        )
    tail = (proc.stderr or proc.stdout or "").strip().splitlines()[-15:]
    raise RuntimeError(f"{label} failed (exit {proc.returncode}):" + chr(10) + chr(10).join(tail))


def prefetch_model() -> None:
    """Download the weights in a child process, for the same reason as the encode."""
    code = (
        "from huggingface_hub import snapshot_download; "
        f"snapshot_download({MODEL_ID!r}); "
        "print('PREFETCH_OK')"
    )
    print("Prefetching model weights (~23 GB on a cold cache; no-op once cached)...", flush=True)
    _run_child([sys.executable, "-c", code], "prefetch")
    ram_report("after prefetch")


prefetch_model()


In [ ]:
import gc
import os
import tempfile
import time

from diffusers.utils import export_to_video

NUM_INFERENCE_STEPS = 25  # Wan's default is 50; 25 roughly halves runtime at a modest quality cost

DEFAULT_NEGATIVE = (
    "overexposed, static, blurred details, subtitles, worst quality, low quality, JPEG artifacts, "
    "ugly, deformed, disfigured, misshapen limbs, fused fingers, still picture, cluttered background, "
    "watermark, text, logo"
)

# Embeddings are a couple of MB; spawning the encoder is the expensive part. A repeat prompt -- a
# retried beat, a re-run of the same story -- skips the subprocess entirely.
_EMBED_CACHE: dict = {}


def _log(msg: str) -> None:
    print(f"[colab_video {time.strftime('%H:%M:%S')}] {msg}", flush=True)


def _free_gpu() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def _round_frames(n: int) -> int:
    """Wan's VAE has temporal stride 4 -> num_frames must be 4k+1."""
    n = max(5, int(n))
    return ((n - 1) // 4) * 4 + 1


def _encode(prompt: str, negative_prompt: str):
    """Embed the prompt in a child process, with the GPU cleared for it first."""
    key = (prompt, negative_prompt)
    if key in _EMBED_CACHE:
        _log("prompt embeddings served from cache")
        return _EMBED_CACHE[key]

    # The child needs ~11.4 GB of VRAM, so the transformer cannot be sitting there.
    if not KEEP_PIPE_ON_GPU:
        pipe.to("cpu")
    _free_gpu()
    ram_report("before encode subprocess")

    out_path = os.path.join(tempfile.gettempdir(), "wan_embeds.pt")
    _log("encoding prompt in a subprocess (loads the 11.4 GB text encoder)...")
    _run_child(
        [
            sys.executable, ENCODE_SCRIPT,
            "--model-id", MODEL_ID,
            "--prompt", prompt,
            "--negative", negative_prompt,
            "--out", out_path,
            "--dtype", DTYPE_NAME,
            "--max-len", str(MAX_SEQUENCE_LENGTH),
            "--device", DEVICE,
        ],
        "encode",
    )

    payload = torch.load(out_path, map_location="cpu")
    os.remove(out_path)
    embeds = (
        payload["prompt_embeds"].to(DEVICE, DTYPE),
        payload["negative_prompt_embeds"].to(DEVICE, DTYPE),
    )
    ram_report("after encode subprocess")

    _EMBED_CACHE[key] = embeds
    return embeds


def generate(prompt: str, negative_prompt: str, num_frames: int, height: int, width: int, guidance_scale: float) -> str:
    num_frames = _round_frames(num_frames)
    height = max(256, int(round(height / 16) * 16))
    width = max(256, int(round(width / 16) * 16))
    neg = (negative_prompt or "").strip() or DEFAULT_NEGATIVE

    _log(f"generate: {num_frames} frames  {width}x{height}  steps={NUM_INFERENCE_STEPS}  gs={guidance_scale}")
    _log(f"prompt: {prompt[:200]}")
    t0 = time.time()
    try:
        prompt_embeds, negative_embeds = _encode(prompt, neg)

        # Only now, with the encoder process gone, is there room for the transformer.
        if not KEEP_PIPE_ON_GPU:
            pipe.to(DEVICE)
            ram_report("pipe on GPU, denoising")

        frames = pipe(
            prompt_embeds=prompt_embeds,
            negative_prompt_embeds=negative_embeds,
            num_frames=num_frames,
            num_inference_steps=NUM_INFERENCE_STEPS,
            guidance_scale=float(guidance_scale),
            height=height,
            width=width,
        ).frames[0]
        _log(f"denoise done in {time.time() - t0:.0f}s -- exporting {len(frames)} frames @ {EXPORT_FPS}fps")
        out_path = tempfile.mktemp(suffix=".mp4")
        export_to_video(frames, out_path, fps=EXPORT_FPS)
        _log(f"wrote {out_path}  (total {time.time() - t0:.0f}s)")
        return out_path
    except Exception as e:
        _log(f"FAILED after {time.time() - t0:.0f}s -- {type(e).__name__}: {e}")
        raise
    finally:
        if not KEEP_PIPE_ON_GPU:
            pipe.to("cpu")
        _free_gpu()


In [ ]:
# One short clip end to end, so a problem surfaces here rather than as a dead endpoint the
# pipeline silently falls back from. 25 frames is ~1.5 s -- a couple of minutes, not ten.
# Watch the [mem] lines: RAM should never approach the total, and the encoder and the transformer
# should never both be on the GPU.
_smoke = generate(
    "a tiny glowing firefly drifting through a dark forest at night, soft golden light, "
    "children's storybook animation style",
    DEFAULT_NEGATIVE,
    25,
    DEFAULT_H,
    DEFAULT_W,
    5.0,
)
print("smoke test OK ->", _smoke)


In [ ]:
import gradio as gr

# Input order MUST match tools/colab_video.py's positional client.predict(...) call exactly:
# prompt, negative_prompt, num_frames, height, width, guidance_scale.
#
# api_name is pinned explicitly: gr.Interface defaults it to "predict" on Gradio <=5.x but to the
# wrapped function's name on 6.x, and the install cell doesn't pin a version. Naming it here makes
# the endpoint "/generate" on every version. (tools/colab_video.py probes both names anyway.)
demo = gr.Interface(
    fn=generate,
    inputs=[
        gr.Textbox(label="prompt", lines=3),
        gr.Textbox(label="negative_prompt", value=DEFAULT_NEGATIVE),
        gr.Number(label="num_frames", value=DEFAULT_FRAMES, precision=0),
        gr.Number(label="height", value=DEFAULT_H, precision=0),
        gr.Number(label="width", value=DEFAULT_W, precision=0),
        gr.Number(label="guidance_scale", value=5.0),
    ],
    outputs=gr.Video(label="output"),
    title=MODEL_ID.split("/")[-1],
    description="Offline AI_VIDEO endpoint for Youtube_video_automation (set COLAB_VIDEO_URL in .env)",
    api_name="generate",
)

demo.launch(share=True, show_error=True)
# Copy the printed https://....gradio.live URL (NOT the localhost one) into .env as COLAB_VIDEO_URL.
